# H4a Nastawienie wobec obozu (NLI)


## 1. Konfiguracja

In [1]:
import sys
from pathlib import Path

if (Path.cwd() / "src").exists():
    sys.path.append(str(Path.cwd()))
else:
    sys.path.append("..")

import pandas as pd
import torch
from tqdm import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer

from src.config import FINETUNED_MODEL_DIR, PROCESSED_DIR
from src.nli import nli_probabilities

H4A_DIR = PROCESSED_DIR / "h4a"
BATCH_SIZE = 8
device = "cuda" if torch.cuda.is_available() else "cpu"
print("urz?dzenie:", device)
if device == "cuda":
    print("CUDA available:", torch.cuda.is_available())


c:\Users\karol\Licencjat_programowanie\Licencjat\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


urz?dzenie: cuda
CUDA available: True


## 2. Wczytanie wzmianek i modelu

In [2]:
mentions = pd.read_parquet(H4A_DIR / "mentions.parquet", columns=["id", "subreddit", "cycle", "term", "text"])
tokenizer = AutoTokenizer.from_pretrained(FINETUNED_MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(FINETUNED_MODEL_DIR, dtype=torch.float32).to(device).eval()
mentions.shape

W1110 00:14:10.982000 7504 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
Loading weights: 100%|██████████| 394/394 [00:00<00:00, 591.10it/s]


(95312, 5)

## 3. Klasyfikacja w pętli po subreddit × cycle

In [3]:
groups = mentions.groupby(["subreddit", "cycle"])
for (subreddit, cycle), group in tqdm(groups, total=groups.ngroups, desc="Klasyfikacja NLI", unit="grupa"):
    group = group.iloc[group["text"].str.len().argsort().values]
    stance = nli_probabilities(group["text"].tolist(), group["term"].tolist(), model, tokenizer, device, BATCH_SIZE)
    stance.insert(0, "id", group["id"].values)
    stance.to_parquet(H4A_DIR / f"stance_{subreddit}_{cycle}.parquet", index=False)

Klasyfikacja NLI: 100%|██████████| 16/16 [5:31:31<00:00, 1243.23s/grupa]  


## 4. Kontrola

In [4]:
stance = pd.concat(pd.read_parquet(path) for path in sorted(H4A_DIR.glob("stance_*.parquet")))
print(len(stance), len(mentions))
stance[["p_entail", "p_entail_3"]].describe()

95312 95312


,p_entail,p_entail_3
count,95312.000000,95312.000000
mean,0.794357,0.763358
std,0.377101,0.391969
min,0.000060,0.000060
25%,0.897319,0.683720
50%,0.998708,0.994922
75%,0.999646,0.998931
max,0.999948,0.999803
